In [1]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U",
     "transformers>=4.51", "accelerate", "sentence-transformers",
     "qdrant-client>=1.10,<2", "pandas", "scikit-learn", "python-dotenv", "tqdm"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
     "--no-deps", "typing_extensions>=4.13"],
    check=True,
)
# torchvision có sẵn trong base image bị lệch ABI với torch sau khi transformers/accelerate
# nâng cấp torch -> lỗi "operator torchvision::nms does not exist". Notebook chỉ cần torch
# (LLM text-only + BGE-M3 text embedding), không xử lý ảnh -> gỡ torchvision/torchaudio.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision", "torchaudio"],
    check=True,
)

print('Cài đặt xong. QUAN TRỌNG: Restart Kernel rồi mới chạy các cell tiếp theo.')



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Cài đặt xong. QUAN TRỌNG: Restart Kernel rồi mới chạy các cell tiếp theo.


In [1]:
import ast
import gc
import hashlib
import json
import platform
import os
import random
import re
import time
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import find_dotenv, load_dotenv
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Tìm .env khi chạy tại root hoặc trong embedding/.
env_path = find_dotenv(usecwd=True)
if not env_path:
    for candidate in [Path('.env'), Path('../.env')]:
        if candidate.exists():
            env_path = str(candidate.resolve())
            break
if env_path:
    load_dotenv(env_path, override=False)
    print('Loaded .env:', env_path)
else:
    print('Không tìm thấy .env; sẽ thử Modal Secret / Kaggle Secrets qua biến môi trường.')

# Mỗi notebook chỉ load một model đầy đủ lên GPU.
# ĐỔI MODEL: sửa đúng 1 dòng dưới đây (giữ nguyên đúng key trong MODEL_REPOS),
# rồi Restart Kernel + Run All. Không cần sửa gì khác để thử model kế tiếp.
AVAILABLE_MODELS = ['DeepSeek-R1-Distill-Llama-8B']
MODEL_REPOS = {
    'Llama-3.1-8B-Instruct': 'meta-llama/Llama-3.1-8B-Instruct',
    'Qwen3-4B': 'Qwen/Qwen3-4B',
    'Qwen2.5-7B-Instruct': 'Qwen/Qwen2.5-7B-Instruct',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    'DeepSeek-R1-Distill-Llama-8B': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B',
    'Llama-3.2-3B': 'meta-llama/Llama-3.2-3B-Instruct',
}

# Chỉ decoding profile được phép khác nhau theo khuyến nghị của nhà sản xuất.
# Mọi retrieval, prompt content, token budget, seed và metric ở dưới đều giống nhau.
MODEL_GENERATION_PROFILES = {
    'Llama-3.1-8B-Instruct': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Llama-3.2-3B': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Qwen2.5-7B-Instruct': {
        'profile_name': 'vendor_qwen2_5_instruct',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.7, 'top_p': 0.8,
            'top_k': 20, 'repetition_penalty': 1.05,
        },
    },
    'Qwen3-4B': {
        'profile_name': 'vendor_qwen3_thinking',
        'enable_thinking': True,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95, 'top_k': 20,
        },
    },
    'DeepSeek-R1-Distill-Qwen-1.5B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
    'DeepSeek-R1-Distill-Llama-8B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
}

assert len(AVAILABLE_MODELS) == 1, 'Mỗi lần chỉ load một model đầy đủ lên GPU.'
MODEL_NAME = AVAILABLE_MODELS[0]
MODEL_ID = MODEL_REPOS[MODEL_NAME]
MODELS_TO_RUN = [MODEL_NAME]
ACTIVE_PROFILE = MODEL_GENERATION_PROFILES[MODEL_NAME]

BENCHMARK_VERSION = 'v2'
BENCHMARK_PROTOCOL = 'vendor_recommended_multi_seed'
BGE_MODEL_ID = 'BAAI/bge-m3'
QDRANT_COLLECTION = 'laws_bge_m3_v2_correct_pooling'
EXPECTED_VECTOR_DIM = 1024
TOP_K = 10
MAX_INPUT_TOKENS = 24000
MAX_NEW_TOKENS = 8192
MAX_ARTICLE_CHARS = 6000  # giới hạn theo từng điều; mọi model nhận cùng chuỗi evidence
MAX_GENERATION_ATTEMPTS = 1  # benchmark strict: không retry để chọn output hợp lệ hơn
FAIL_FAST = False
INVALID_OUTPUT_LABEL = '__INVALID_OUTPUT__'
EVAL_SEEDS = [2026]
OUTPUT_DIR = Path('outputs_alqac_e2e') / BENCHMARK_VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ['A_WIN', 'B_WIN', 'PARTIAL_A_WIN', 'PARTIAL_B_WIN']
assert MAX_GENERATION_ATTEMPTS == 1
assert len(EVAL_SEEDS) == len(set(EVAL_SEEDS))
random.seed(EVAL_SEEDS[0])
np.random.seed(EVAL_SEEDS[0])

def get_secret(*names, required=True):
    # Modal Notebook: secret được attach lúc tạo notebook -> đã có sẵn trong os.environ,
    # không cần bước nào khác. Fallback Kaggle Secrets chỉ kích hoạt khi chạy trên Kaggle.
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in names:
            try:
                value = client.get_secret(name)
                if value:
                    return value
            except Exception:
                pass
    except Exception:
        pass
    if required:
        raise RuntimeError(f'Thiếu secret, cần một trong: {names}')
    return None

HF_TOKEN = get_secret('HF_TOKEN', required=False)
QDRANT_URL = get_secret('QDRANT_URL')
QDRANT_API_KEY = get_secret('QDRANT_API_KEY', 'QDRANT_KEY')

assert torch.cuda.is_available(), 'Notebook này yêu cầu GPU CUDA — khi tạo Modal Notebook nhớ chọn GPU (A10G trở lên cho model 7-8B).'
torch.manual_seed(EVAL_SEEDS[0])
torch.cuda.manual_seed_all(EVAL_SEEDS[0])
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('GPU:', torch.cuda.get_device_name(0))
print('Model:', MODEL_NAME, '->', MODEL_ID)
print('Benchmark:', BENCHMARK_VERSION, '| protocol:', BENCHMARK_PROTOCOL)
print('Generation profile:', ACTIVE_PROFILE['profile_name'], '| seeds:', EVAL_SEEDS)
print('Precision:', 'unquantized', 'BF16' if torch.cuda.is_bf16_supported() else 'FP16')


Không tìm thấy .env; sẽ thử Modal Secret / Kaggle Secrets qua biến môi trường.
GPU: NVIDIA L4
Model: DeepSeek-R1-Distill-Llama-8B -> deepseek-ai/DeepSeek-R1-Distill-Llama-8B
Benchmark: v2 | protocol: vendor_recommended_multi_seed
Generation profile: vendor_deepseek_r1_distill | seeds: [2026]
Precision: unquantized BF16


In [3]:
def find_public_test():
    # Có thể override mà không sửa notebook: ALQAC_PUBLIC_TEST_PATH=/path/to/file.json
    candidates = []
    if os.getenv('ALQAC_PUBLIC_TEST_PATH'):
        candidates.append(Path(os.environ['ALQAC_PUBLIC_TEST_PATH']))
    candidates += [
        # Modal Notebook: upload ALQAC2026_public_test.json qua file browser bên trái
        # (kéo thả vào đúng thư mục làm việc của notebook) -> sẽ khớp 1 trong 2 dòng dưới.
        Path('ALQAC2026_public_test.json'),
        Path('data/ALQAC2026_public_test.json'),
        Path('../data/ALQAC2026_public_test.json'),
        # Kaggle (giữ lại để notebook vẫn chạy được trên Kaggle nếu cần đối chiếu).
        Path('/kaggle/input/datasets/ldhhieu18/demnguoctoibinhminh/ALQAC2026_public_test.json'),
        Path('/kaggle/working/ALQAC2026_public_test.json'),
    ]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob('ALQAC2026_public_test.json'))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    checked = '\n'.join(f'  - {path}' for path in candidates)
    raise FileNotFoundError(
        f'Không tìm thấy ALQAC2026_public_test.json. Đã kiểm tra:\n{checked}\n'
        f'Trên Modal Notebook: upload file này qua file browser, hoặc set '
        f"os.environ['ALQAC_PUBLIC_TEST_PATH'] ở cell trước khi gọi find_public_test()."
    )

DATA_PATH = find_public_test()
with DATA_PATH.open(encoding='utf-8') as f:
    public_data = json.load(f)

assert len(public_data) == 50, f'Expected 50 cases, got {len(public_data)}'
assert len({x['case_id'] for x in public_data}) == len(public_data)
assert all(x.get('case_query') for x in public_data)
assert all(x.get('verdict_label') in LABELS for x in public_data)

# Đây là view duy nhất được pipeline dự đoán sử dụng. Gold được giữ riêng cho cell đánh giá.
inference_cases = [
    {'case_id': x['case_id'], 'case_query': x['case_query']}
    for x in public_data
]
gold_by_case = {x['case_id']: x['verdict_label'] for x in public_data}

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=120)
if not qdrant.collection_exists(QDRANT_COLLECTION):
    raise RuntimeError(
        f'Chưa có collection {QDRANT_COLLECTION}. Hãy chạy notebook BGE-M3 trước.'
    )
collection_info = qdrant.get_collection(QDRANT_COLLECTION)
stored_dim = int(collection_info.config.params.vectors.size)
stored_count = int(qdrant.count(QDRANT_COLLECTION, exact=True).count)
assert stored_dim == EXPECTED_VECTOR_DIM, (stored_dim, EXPECTED_VECTOR_DIM)
assert stored_count == 3352, f'Expected 3352 laws, got {stored_count}'

embedder = SentenceTransformer(
    BGE_MODEL_ID, device='cuda' if torch.cuda.is_available() else 'cpu'
)
embedder.max_seq_length = 8192

print('Dataset:', DATA_PATH)
print('Cases:', len(inference_cases))
print('Qdrant:', QDRANT_COLLECTION, '| dim:', stored_dim, '| points:', stored_count)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Dataset: /root/ALQAC2026_public_test.json
Cases: 50
Qdrant: laws_bge_m3_v2_correct_pooling | dim: 1024 | points: 3352


In [4]:
def retrieve_laws(case_query, top_k=TOP_K):
    query_vector = embedder.encode(
        [case_query], normalize_embeddings=True, convert_to_numpy=True
    )[0]
    assert query_vector.shape == (EXPECTED_VECTOR_DIM,)
    assert np.isfinite(query_vector).all()
    hits = qdrant.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector.tolist(),
        limit=top_k,
        with_payload=True,
    ).points
    laws = []
    seen = set()
    for rank, hit in enumerate(hits, 1):
        payload = hit.payload or {}
        key = (str(payload.get('law_id')), int(payload.get('aid')))
        if key in seen:
            continue
        seen.add(key)
        laws.append({
            'rank': rank,
            'score': float(hit.score),
            'law_id': key[0],
            'aid': key[1],
            'article_no': int(payload.get('article_no')),
            'content_Article': str(payload.get('content_Article') or ''),
        })
    assert len(laws) == top_k, f'Chỉ retrieve được {len(laws)}/{top_k} luật'
    return laws

sample_case = inference_cases[0]
sample_laws = retrieve_laws(sample_case['case_query'])
display(pd.DataFrame(sample_laws)[['rank', 'score', 'law_id', 'article_no', 'aid']])

# Retrieve đủ 50 case trước, rồi giải phóng BGE-M3 để dành toàn bộ VRAM cho LLM full precision.
retrieval_path = OUTPUT_DIR / 'retrieval_top10_bge_m3.json'
retrieval_meta_path = OUTPUT_DIR / 'retrieval_top10_bge_m3.meta.json'
retrieval_signature_material = {
    'bge_model': BGE_MODEL_ID, 'collection': QDRANT_COLLECTION, 'top_k': TOP_K,
    'cases': inference_cases,
}
retrieval_signature = hashlib.sha256(
    json.dumps(retrieval_signature_material, ensure_ascii=False, sort_keys=True).encode('utf-8')
).hexdigest()
cached_signature = None
if retrieval_meta_path.exists():
    try:
        cached_signature = json.loads(retrieval_meta_path.read_text(encoding='utf-8')).get('signature')
    except Exception:
        pass
if retrieval_path.exists() and cached_signature == retrieval_signature:
    try:
        retrieval_cache = json.loads(retrieval_path.read_text(encoding='utf-8'))
    except Exception:
        retrieval_cache = {}
else:
    retrieval_cache = {}
for case in tqdm(inference_cases, desc='BGE-M3 retrieval'):
    if case['case_id'] not in retrieval_cache:
        retrieval_cache[case['case_id']] = retrieve_laws(case['case_query'])
        temp_path = retrieval_path.with_suffix('.json.tmp')
        temp_path.write_text(json.dumps(retrieval_cache, ensure_ascii=False, indent=2), encoding='utf-8')
        temp_path.replace(retrieval_path)
retrieval_meta_path.write_text(
    json.dumps({'signature': retrieval_signature, **retrieval_signature_material}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
expected_case_ids = {x['case_id'] for x in inference_cases}
assert set(retrieval_cache) == expected_case_ids
assert all(len(retrieval_cache[cid]) == TOP_K for cid in expected_case_ids)

del embedder
gc.collect()
torch.cuda.empty_cache()
print('Đã cache retrieval và giải phóng BGE-M3. GPU allocated:', round(torch.cuda.memory_allocated()/2**30, 2), 'GB')


,rank,score,law_id,article_no,aid
0,1,0.592945,91/2015/QH13,603,53373
1,2,0.513985,26/2008/QH12,10,5275
2,3,0.506477,91/2015/QH13,13,52783
3,4,0.504947,91/2015/QH13,687,53457
4,5,0.500724,91/2015/QH13,584,53354
5,6,0.498698,91/2015/QH13,600,53370
6,7,0.497034,91/2015/QH13,583,53353
7,8,0.494676,92/2015/QH13,206,50871
8,9,0.493694,91/2015/QH13,605,53375
9,10,0.493415,91/2015/QH13,588,53358


BGE-M3 retrieval:   0%|          | 0/50 [00:00<?, ?it/s]

Đã cache retrieval và giải phóng BGE-M3. GPU allocated: 0.01 GB


In [ ]:
# Load đầy đủ model lên GPU, không quantization và không CPU offload.
MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=MODEL_DTYPE,
    device_map={'': 0},
    low_cpu_mem_usage=True,
)
llm_model.eval()
model_devices = {p.device.type for p in llm_model.parameters()}
assert model_devices == {'cuda'}, f'Model chưa nằm hoàn toàn trên GPU: {model_devices}'
MODEL_REVISION = getattr(llm_model.config, '_commit_hash', None) or 'unknown'
RESOLVED_GENERATION_CONFIG = llm_model.generation_config.to_dict()
print(
    'Loaded unquantized model:', MODEL_ID,
    '| dtype:', MODEL_DTYPE,
    '| GPU allocated:', round(torch.cuda.memory_allocated()/2**30, 2), 'GB',
)
print('Model revision:', MODEL_REVISION)
print('Resolved model generation config:', json.dumps(RESOLVED_GENERATION_CONFIG, ensure_ascii=False))

SYSTEM_PROMPT = '''Bạn là chuyên gia phân tích tranh chấp dân sự Việt Nam.
Bạn chỉ được sử dụng CASE_QUERY và 10 ĐIỀU LUẬT được cung cấp. Không được giả định dữ kiện ngoài đầu vào.
A là nguyên đơn, B là bị đơn. Hãy dự đoán đúng một trong bốn nhãn:
- A_WIN: toàn bộ hoặc về cơ bản toàn bộ yêu cầu của nguyên đơn được chấp nhận.
- B_WIN: yêu cầu của nguyên đơn bị bác toàn bộ hoặc về cơ bản toàn bộ.
- PARTIAL_A_WIN: nguyên đơn được chấp nhận một phần đáng kể nhưng không toàn bộ; kết quả nghiêng về A.
- PARTIAL_B_WIN: có phần yêu cầu của nguyên đơn được chấp nhận nhưng kết quả chủ yếu nghiêng về B.

Trả về đúng một JSON object, không Markdown, không văn bản bên ngoài JSON:
{
  "prediction": "<LABEL>",
  "confidence": 0.78,
  "reasoning": "Lập luận ngắn gọn bằng tiếng Việt",
  "applied_laws": [
    {"law_id": "91/2015/QH13", "aid": 53373, "reason": "Lý do áp dụng"}
  ]
}
Thay <LABEL> bằng đúng một trong A_WIN, B_WIN, PARTIAL_A_WIN, PARTIAL_B_WIN; không được giữ placeholder.
Chỉ chọn applied_laws từ danh sách 10 điều luật. Confidence phải nằm trong [0, 1].'''

BENCHMARK_MANIFEST = {
    'benchmark_version': BENCHMARK_VERSION,
    'benchmark_protocol': BENCHMARK_PROTOCOL,
    'model_name': MODEL_NAME, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'generation_profile': ACTIVE_PROFILE,
    'resolved_model_generation_config': RESOLVED_GENERATION_CONFIG,
    'dtype': str(MODEL_DTYPE), 'full_gpu_no_quantization': True,
    'gpu': torch.cuda.get_device_name(0),
    'gpu_total_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    'python': platform.python_version(), 'torch': torch.__version__,
    'transformers': package_version('transformers'),
    'sentence_transformers': package_version('sentence-transformers'),
    'dataset_path': str(DATA_PATH),
    'dataset_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'num_cases': len(inference_cases), 'labels': LABELS,
    'bge_model': BGE_MODEL_ID, 'qdrant_collection': QDRANT_COLLECTION, 'top_k_laws': TOP_K,
    'max_input_tokens': MAX_INPUT_TOKENS, 'max_new_tokens': MAX_NEW_TOKENS,
    'max_article_chars': MAX_ARTICLE_CHARS,
    'max_generation_attempts': MAX_GENERATION_ATTEMPTS,
    'eval_seeds': EVAL_SEEDS,
    'invalid_output_policy': 'count_as_wrong',
    'system_prompt_sha256': hashlib.sha256(SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
    'retrieval_signature': retrieval_signature,
}

def build_user_prompt(case_query, laws):
    law_blocks = []
    for law in laws:
        content = law['content_Article'][:MAX_ARTICLE_CHARS]
        law_blocks.append(
            f"[{law['rank']}] law_id={law['law_id']} | Điều {law['article_no']} | aid={law['aid']}\n"
            f"{content}"
        )
    return (
        'CASE_QUERY:\n' + case_query.strip() +
        '\n\n10 ĐIỀU LUẬT TRUY XUẤT:\n' + '\n\n'.join(law_blocks) +
        '\n\nHãy phân tích và trả về đúng JSON schema đã yêu cầu.'
    )

def build_messages(user_prompt):
    if ACTIVE_PROFILE['use_system_prompt']:
        return [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ]
    # DeepSeek-R1 khuyến nghị không dùng system role; nội dung hướng dẫn vẫn giữ nguyên.
    return [{'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + user_prompt}]

def extract_first_json(text):
    text = re.sub(r'<think>.*?</think>', '', text or '', flags=re.I | re.S).strip()
    if '</think>' in text:
        text = text.rsplit('</think>', 1)[-1].strip()
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.I | re.S).strip()
    decoder = json.JSONDecoder()
    for match in re.finditer(r'\{', text):
        try:
            obj, _ = decoder.raw_decode(text[match.start():])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue
    try:
        obj = ast.literal_eval(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    raise ValueError('Không tìm thấy JSON object hợp lệ')

def validate_prediction(obj, retrieved_laws):
    prediction = str(obj.get('prediction', '')).strip().upper()
    if prediction not in LABELS:
        raise ValueError(f'prediction không hợp lệ: {prediction!r}')
    confidence = float(obj.get('confidence'))
    if not 0.0 <= confidence <= 1.0:
        raise ValueError(f'confidence ngoài [0,1]: {confidence}')
    reasoning = str(obj.get('reasoning', '')).strip()
    if not reasoning:
        raise ValueError('reasoning rỗng')
    allowed = {(x['law_id'], int(x['aid'])) for x in retrieved_laws}
    clean_laws = []
    seen = set()
    for item in obj.get('applied_laws', []):
        try:
            key = (str(item['law_id']), int(item['aid']))
        except Exception:
            continue
        if key not in allowed or key in seen:
            continue
        seen.add(key)
        clean_laws.append({
            'law_id': key[0], 'aid': key[1],
            'reason': str(item.get('reason', '')).strip(),
        })
    return {
        'prediction': prediction,
        'confidence': confidence,
        'reasoning': reasoning,
        'applied_laws': clean_laws,
    }

class PredictionFormatError(RuntimeError):
    def __init__(self, message, raw_response, usage):
        super().__init__(message)
        self.raw_response = raw_response
        self.usage = usage

def call_local_llm(model_name, case_query, retrieved_laws, eval_seed):
    assert model_name == MODEL_NAME
    assert eval_seed in EVAL_SEEDS
    user_prompt = build_user_prompt(case_query, retrieved_laws)
    messages = build_messages(user_prompt)
    template_kwargs = {}
    if 'qwen3' in MODEL_ID.lower():
        template_kwargs['enable_thinking'] = ACTIVE_PROFILE['enable_thinking']
    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, **template_kwargs
    )
    # Không truncation âm thầm: input vượt budget sẽ thành output lỗi có thể kiểm toán.
    inputs = tokenizer(rendered, return_tensors='pt', truncation=False)
    input_tokens = int(inputs['input_ids'].shape[1])
    if input_tokens > MAX_INPUT_TOKENS:
        raise ValueError(f'Input vượt budget: {input_tokens}/{MAX_INPUT_TOKENS} tokens')
    input_device = llm_model.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    case_seed = eval_seed + int(hashlib.sha256(case_query.encode('utf-8')).hexdigest()[:8], 16)
    torch.manual_seed(case_seed)
    torch.cuda.manual_seed_all(case_seed)
    generation_kwargs = {
        'max_new_tokens': MAX_NEW_TOKENS,
        'pad_token_id': tokenizer.pad_token_id,
        'use_cache': True,
        **ACTIVE_PROFILE['generation_kwargs'],
    }
    try:
        with torch.inference_mode():
            generated = llm_model.generate(**inputs, **generation_kwargs)
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        raise RuntimeError(
            'CUDA OOM: giảm MAX_INPUT_TOKENS/MAX_NEW_TOKENS hoặc dùng GPU VRAM lớn hơn.'
        ) from exc

    output_ids = generated[0, input_tokens:]
    output_tokens = int(output_ids.numel())
    raw_text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    usage = {
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': input_tokens + output_tokens,
        'hit_max_new_tokens': output_tokens >= MAX_NEW_TOKENS,
        'eval_seed': eval_seed,
        'case_seed': case_seed,
    }
    del generated, output_ids
    if not raw_text:
        raise PredictionFormatError('Model trả output rỗng', raw_text, usage)
    try:
        parsed = validate_prediction(extract_first_json(raw_text), retrieved_laws)
    except Exception as exc:
        raise PredictionFormatError(str(exc), raw_text, usage) from exc
    return parsed, raw_text, usage, user_prompt


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Loaded unquantized model: deepseek-ai/DeepSeek-R1-Distill-Llama-8B | dtype: torch.bfloat16 | GPU allocated: 14.97 GB
Model revision: 6a6f4aa4197940add57724a7707d069478df56b1
Resolved model generation config: {"max_length": null, "max_new_tokens": null, "min_length": null, "min_new_tokens": null, "early_stopping": null, "max_time": null, "stop_strings": null, "do_sample": true, "num_beams": null, "use_mtp": null, "use_cache": null, "cache_implementation": null, "cache_config": null, "max_cache_len": null, "temperature": 0.6, "top_k": null, "top_p": 0.95, "min_p": null, "top_h": null, "typical_p": null, "epsilon_cutoff": null, "eta_cutoff": null, "repetition_penalty": null, "encoder_repetition_penalty": null, "length_penalty": null, "no_repeat_ngram_size": null, "bad_words_ids": null, "renormalize_logits": null, "forced_bos_token_id": null, "forced_eos_token_id": null, "remove_invalid_values": null, "exponential_decay_length_penalty": null, "suppress_tokens": null, "begin_suppress_tokens

In [6]:
smoke_model = MODELS_TO_RUN[0]
smoke_seed = EVAL_SEEDS[0]
smoke_laws = retrieval_cache[sample_case['case_id']]
try:
    smoke_result, smoke_raw, smoke_usage, _ = call_local_llm(
        smoke_model, sample_case['case_query'], smoke_laws, smoke_seed
    )
    print('Model:', smoke_model, '| seed:', smoke_seed)
    print(json.dumps(smoke_result, ensure_ascii=False, indent=2))
    print('Usage:', smoke_usage)
except Exception as exc:
    # Smoke lỗi không làm dừng benchmark; batch vẫn chấm case này đúng một lần theo seed.
    print('SMOKE WARNING:', repr(exc))


Model: DeepSeek-R1-Distill-Llama-8B | seed: 2026
{
  "prediction": "PARTIAL_A_WIN",
  "confidence": 0.78,
  "reasoning": "Chị T là nguyên đơn, có yêu cầu bồi thường thiệt hại do chó nhà anh D thả rông gây ra. Điều 603 quy định chủ sở hữu súc vật phải bồi thường thiệt hại cho người khác. Anh D là người chiếm hữu chó, phải chịu trách nhiệm bồi thường theo Điều 603. Yêu cầu của chị T về bồi thường chi phí sửa xe và viện phí được chấp nhận một phần, nhưng kết quả nghiêng về A.",
  "applied_laws": [
    {
      "law_id": "91/2015/QH13",
      "aid": 53373,
      "reason": "Chủ sở hữu súc vật phải bồi thường thiệt hại do súc vật gây ra cho người khác."
    }
  ]
}
Usage: {'input_tokens': 2035, 'output_tokens': 480, 'total_tokens': 2515, 'hit_max_new_tokens': False, 'eval_seed': 2026, 'case_seed': 743951647}


In [7]:
def slugify(text):
    return re.sub(r'[^a-z0-9]+', '-', text.lower()).strip('-')

def load_json(path, default):
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return default

def atomic_write_json(path, obj):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)

def make_cache_key(model, case, laws, eval_seed):
    material = {
        'benchmark_version': BENCHMARK_VERSION,
        'benchmark_protocol': BENCHMARK_PROTOCOL,
        'model': model, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
        'generation_profile': ACTIVE_PROFILE,
        'resolved_model_generation_config': RESOLVED_GENERATION_CONFIG,
        'eval_seed': eval_seed,
        'case_id': case['case_id'],
        'case_query': case['case_query'], 'laws': laws,
        'system_prompt': SYSTEM_PROMPT,
        'max_input_tokens': MAX_INPUT_TOKENS,
        'max_new_tokens': MAX_NEW_TOKENS,
        'max_generation_attempts': MAX_GENERATION_ATTEMPTS,
    }
    raw = json.dumps(material, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()

manifest_path = OUTPUT_DIR / f'benchmark_manifest_{slugify(MODEL_NAME)}.json'
atomic_write_json(manifest_path, BENCHMARK_MANIFEST)
print('Benchmark manifest:', manifest_path)

assert len(retrieval_cache) == len(inference_cases)

# all_model_results[model][seed][case_id] -> result
all_model_results = {}
for model in MODELS_TO_RUN:
    all_model_results[model] = {}
    for eval_seed in EVAL_SEEDS:
        result_path = OUTPUT_DIR / f'predictions_{slugify(model)}_seed-{eval_seed}.json'
        saved = load_json(result_path, {})
        print(f'\n=== {model} | seed={eval_seed} | cached {len(saved)}/{len(inference_cases)} ===')
        for case in tqdm(inference_cases, desc=f'{model} seed={eval_seed}'):
            case_id = case['case_id']
            laws = retrieval_cache[case_id]
            cache_key = make_cache_key(model, case, laws, eval_seed)
            old = saved.get(case_id)
            # Cache cả output lỗi: không cho case thêm cơ hội chỉ vì lần trước sai format.
            if old and old.get('cache_key') == cache_key:
                continue
            started = time.time()
            try:
                parsed, raw_text, usage, _ = call_local_llm(
                    model, case['case_query'], laws, eval_seed
                )
                saved[case_id] = {
                    'case_id': case_id,
                    'case_query': case['case_query'],
                    'eval_seed': eval_seed,
                    **parsed,
                    'retrieved_laws': laws,
                    'raw_response': raw_text,
                    'usage': usage,
                    'duration_seconds': round(time.time() - started, 3),
                    'generation_attempts': 1,
                    'cache_key': cache_key,
                    'error': None,
                }
            except Exception as exc:
                saved[case_id] = {
                    'case_id': case_id,
                    'case_query': case['case_query'],
                    'eval_seed': eval_seed,
                    'prediction': None,
                    'confidence': None,
                    'reasoning': '',
                    'applied_laws': [],
                    'retrieved_laws': laws,
                    'raw_response': getattr(exc, 'raw_response', None),
                    'usage': getattr(exc, 'usage', None),
                    'duration_seconds': round(time.time() - started, 3),
                    'generation_attempts': 1,
                    'cache_key': cache_key,
                    'error': repr(exc),
                }
                print(f'  INVALID {case_id} | seed={eval_seed}: {exc}')
                if FAIL_FAST:
                    atomic_write_json(result_path, saved)
                    raise
            atomic_write_json(result_path, saved)
        all_model_results[model][eval_seed] = saved

print('Hoàn tất batch cho', len(EVAL_SEEDS), 'seed x', len(inference_cases), 'case.')


Benchmark manifest: outputs_alqac_e2e/v2/benchmark_manifest_deepseek-r1-distill-llama-8b.json

=== DeepSeek-R1-Distill-Llama-8B | seed=2026 | cached 0/50 ===


DeepSeek-R1-Distill-Llama-8B seed=2026:   0%|          | 0/50 [00:00<?, ?it/s]

  INVALID case_8732 | seed=2026: Không tìm thấy JSON object hợp lệ
Hoàn tất batch cho 1 seed x 50 case.


In [8]:
def evaluate_run(model, eval_seed, result_map):
    rows = []
    for case in inference_cases:
        cid = case['case_id']
        item = result_map.get(cid, {})
        prediction = item.get('prediction')
        is_valid_output = prediction in LABELS
        usage = item.get('usage') or {}
        rows.append({
            'model': model,
            'seed': eval_seed,
            'case_id': cid,
            'gold': gold_by_case[cid],
            'prediction': prediction,
            'scored_prediction': prediction if is_valid_output else INVALID_OUTPUT_LABEL,
            'is_valid_output': is_valid_output,
            'confidence': item.get('confidence'),
            'input_tokens': usage.get('input_tokens'),
            'output_tokens': usage.get('output_tokens'),
            'hit_max_new_tokens': bool(usage.get('hit_max_new_tokens', False)),
            'duration_seconds': item.get('duration_seconds'),
            'error': item.get('error'),
        })
    frame = pd.DataFrame(rows)
    valid = frame[frame['is_valid_output']].copy()
    n_total, n_valid = len(frame), len(valid)
    n_failed = n_total - n_valid
    if n_failed:
        failed_ids = frame.loc[~frame['is_valid_output'], 'case_id'].tolist()
        print(
            f'Cảnh báo {model} seed={eval_seed}: {n_failed}/{n_total} output không hợp lệ '
            f'được tính sai. Case: {failed_ids}'
        )

    scored_prediction = frame['scored_prediction']
    strict_correct = int((frame['gold'] == scored_prediction).sum())
    strict_accuracy = strict_correct / n_total if n_total else 0.0
    valid_accuracy = accuracy_score(valid['gold'], valid['prediction']) if n_valid else 0.0
    report = classification_report(
        frame['gold'], scored_prediction, labels=LABELS,
        output_dict=True, zero_division=0,
    ) if n_total else {}
    cm_all = confusion_matrix(
        frame['gold'], scored_prediction, labels=LABELS + [INVALID_OUTPUT_LABEL]
    ) if n_total else np.zeros((len(LABELS) + 1, len(LABELS) + 1), dtype=int)
    cm = cm_all[:len(LABELS), :]
    summary = {
        'benchmark_version': BENCHMARK_VERSION,
        'benchmark_protocol': BENCHMARK_PROTOCOL,
        'model': model,
        'seed': eval_seed,
        'n_total': n_total,
        'n_success': n_valid,
        'n_failed': n_failed,
        'invalid_output_rate': n_failed / n_total if n_total else 0.0,
        'coverage': n_valid / n_total if n_total else 0.0,
        'benchmark_valid': n_total == len(inference_cases),
        'all_outputs_valid': n_valid == n_total,
        'metric_scope': 'all_50_invalid_outputs_count_as_wrong',
        'strict_accuracy_all_50': strict_accuracy,
        'accuracy_successful_only': valid_accuracy,
        'macro_precision': report.get('macro avg', {}).get('precision', 0.0),
        'macro_recall': report.get('macro avg', {}).get('recall', 0.0),
        'macro_f1': report.get('macro avg', {}).get('f1-score', 0.0),
        'weighted_f1': report.get('weighted avg', {}).get('f1-score', 0.0),
        'avg_input_tokens': frame['input_tokens'].mean(),
        'avg_output_tokens': frame['output_tokens'].mean(),
        'avg_duration_seconds': frame['duration_seconds'].mean(),
        'n_hit_max_new_tokens': int(frame['hit_max_new_tokens'].sum()),
    }
    per_label = pd.DataFrame([
        {
            'model': model,
            'seed': eval_seed,
            'label': label,
            'precision': report.get(label, {}).get('precision', 0.0),
            'recall': report.get(label, {}).get('recall', 0.0),
            'f1': report.get(label, {}).get('f1-score', 0.0),
            'support': int(report.get(label, {}).get('support', 0)),
        } for label in LABELS
    ])
    cm_frame = pd.DataFrame(
        cm,
        index=[f'gold_{x}' for x in LABELS],
        columns=[f'pred_{x}' for x in LABELS + [INVALID_OUTPUT_LABEL]],
    )
    return summary, per_label, cm_frame, frame

summaries = []
evaluation_artifacts = {}
for model, seed_results in all_model_results.items():
    evaluation_artifacts[model] = {}
    for eval_seed, results in seed_results.items():
        summary, per_label, cm_frame, case_frame = evaluate_run(model, eval_seed, results)
        summaries.append(summary)
        evaluation_artifacts[model][eval_seed] = {
            'per_label': per_label,
            'confusion_matrix': cm_frame,
            'cases': case_frame,
        }
        print(f'\n=== {model} | seed={eval_seed} ===')
        display(pd.DataFrame([summary]))
        display(cm_frame)

run_metrics = pd.DataFrame(summaries).sort_values(['model', 'seed']).reset_index(drop=True)
aggregate_metrics = run_metrics.groupby('model', as_index=False).agg(
    n_seeds=('seed', 'nunique'),
    accuracy_mean=('strict_accuracy_all_50', 'mean'),
    accuracy_std=('strict_accuracy_all_50', 'std'),
    macro_f1_mean=('macro_f1', 'mean'),
    macro_f1_std=('macro_f1', 'std'),
    invalid_rate_mean=('invalid_output_rate', 'mean'),
    invalid_rate_std=('invalid_output_rate', 'std'),
    avg_output_tokens=('avg_output_tokens', 'mean'),
    avg_duration_seconds=('avg_duration_seconds', 'mean'),
)
aggregate_metrics[['accuracy_std', 'macro_f1_std', 'invalid_rate_std']] = (
    aggregate_metrics[['accuracy_std', 'macro_f1_std', 'invalid_rate_std']].fillna(0.0)
)
leaderboard = aggregate_metrics.sort_values(
    ['accuracy_mean', 'macro_f1_mean'], ascending=False
).reset_index(drop=True)

majority_label = pd.Series(list(gold_by_case.values())).value_counts().idxmax()
majority_accuracy = pd.Series(list(gold_by_case.values())).value_counts().max() / len(gold_by_case)
print(f'Majority baseline: {majority_label} | accuracy={majority_accuracy:.4f}')
print('Per-seed metrics:')
display(run_metrics)
print('Aggregate mean ± std across seeds:')
display(leaderboard)

for model, seed_artifacts in evaluation_artifacts.items():
    slug = slugify(model)
    model_runs = run_metrics[run_metrics['model'] == model]
    model_summary = leaderboard[leaderboard['model'] == model]
    model_runs.to_csv(
        OUTPUT_DIR / f'model_metrics_by_seed_{slug}.csv', index=False, encoding='utf-8-sig'
    )
    model_summary.to_csv(
        OUTPUT_DIR / f'model_metrics_summary_{slug}.csv', index=False, encoding='utf-8-sig'
    )
    for eval_seed, artifacts in seed_artifacts.items():
        suffix = f'{slug}_seed-{eval_seed}'
        artifacts['per_label'].to_csv(
            OUTPUT_DIR / f'metrics_per_label_{suffix}.csv', index=False, encoding='utf-8-sig'
        )
        artifacts['confusion_matrix'].to_csv(
            OUTPUT_DIR / f'confusion_matrix_{suffix}.csv', encoding='utf-8-sig'
        )
        artifacts['cases'].to_csv(
            OUTPUT_DIR / f'case_predictions_{suffix}.csv', index=False, encoding='utf-8-sig'
        )


Cảnh báo DeepSeek-R1-Distill-Llama-8B seed=2026: 1/50 output không hợp lệ được tính sai. Case: ['case_8732']

=== DeepSeek-R1-Distill-Llama-8B | seed=2026 ===


,benchmark_version,benchmark_protocol,model,seed,n_total,n_success,n_failed,invalid_output_rate,coverage,benchmark_valid,...,strict_accuracy_all_50,accuracy_successful_only,macro_precision,macro_recall,macro_f1,weighted_f1,avg_input_tokens,avg_output_tokens,avg_duration_seconds,n_hit_max_new_tokens
0,v2,vendor_recommended_multi_seed,DeepSeek-R1-Distill-Llama-8B,2026,50,49,1,0.02,0.98,True,...,0.26,0.265306,0.200549,0.226316,0.194588,0.226999,3474.74,1110.5,77.42034,1


,pred_A_WIN,pred_B_WIN,pred_PARTIAL_A_WIN,pred_PARTIAL_B_WIN,pred___INVALID_OUTPUT__
gold_A_WIN,8,6,1,1,0
gold_B_WIN,5,3,2,0,0
gold_PARTIAL_A_WIN,12,4,2,0,1
gold_PARTIAL_B_WIN,3,0,2,0,0


Majority baseline: PARTIAL_A_WIN | accuracy=0.3800
Per-seed metrics:


,benchmark_version,benchmark_protocol,model,seed,n_total,n_success,n_failed,invalid_output_rate,coverage,benchmark_valid,...,strict_accuracy_all_50,accuracy_successful_only,macro_precision,macro_recall,macro_f1,weighted_f1,avg_input_tokens,avg_output_tokens,avg_duration_seconds,n_hit_max_new_tokens
0,v2,vendor_recommended_multi_seed,DeepSeek-R1-Distill-Llama-8B,2026,50,49,1,0.02,0.98,True,...,0.26,0.265306,0.200549,0.226316,0.194588,0.226999,3474.74,1110.5,77.42034,1


Aggregate mean ± std across seeds:


,model,n_seeds,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,invalid_rate_mean,invalid_rate_std,avg_output_tokens,avg_duration_seconds
0,DeepSeek-R1-Distill-Llama-8B,1,0.26,0.0,0.194588,0.0,0.02,0.0,1110.5,77.42034


In [9]:
for model, seed_results in all_model_results.items():
    for eval_seed, results in seed_results.items():
        submission = []
        for case in inference_cases:
            item = results.get(case['case_id'], {})
            if item.get('prediction') not in LABELS:
                continue
            submission.append({
                'case_id': case['case_id'],
                'prediction': item['prediction'],
                'case_evidence': [],
                'law_evidence': [
                    {'law_id': law['law_id'], 'aid': int(law['aid'])}
                    for law in item.get('applied_laws', [])
                ],
            })
        path = OUTPUT_DIR / f'submission_{slugify(model)}_seed-{eval_seed}.json'
        atomic_write_json(path, submission)
        n_failed = len(inference_cases) - len(submission)
        print(
            model,
            '| seed:', eval_seed,
            '| evaluated:', len(inference_cases), '/ 50',
            '| invalid counted wrong:', n_failed,
            '| valid submission rows:', len(submission), '/ 50',
            '|', path,
        )

print('Outputs:', OUTPUT_DIR.resolve())


DeepSeek-R1-Distill-Llama-8B | seed: 2026 | evaluated: 50 / 50 | invalid counted wrong: 1 | valid submission rows: 49 / 50 | outputs_alqac_e2e/v2/submission_deepseek-r1-distill-llama-8b_seed-2026.json
Outputs: /root/outputs_alqac_e2e/v2
